# Benchmark Quantum Fourier Transform

The Quantum Fourier Transform (QFT) function is a quantum algorithmic primitive, utilized algorithms such as Shor's Factoring, Discrete Logarithm  and Quantum Phase Estimation. It is the quantum analog for
discrete Fourier transform, applied on the quantum variable state vector.

The basis states $|x\rangle_n$ on $n$ qubits, where $x\in\{0,\dots, N-1\}$ and $N=2^n$, transform as

$${\cal{F}}_N|x\rangle = \frac{1}{\sqrt{N}} \sum_{x=0}^{N-1} e^{2\pi i x k/N}| k\rangle ~~.$$

Therefore, for a general state $|\psi\rangle = \sum_{x=0}^{N-1}a_x |x\rangle$,  the components of $|\chi\rangle  = {\cal{F}_N}|\psi \rangle= \sum_{k=0}^{N-1}b_k |k \rangle$.
satisfy $$b_k =\frac{1}{\sqrt{N}} \sum_{x=0}^{N-1}a_x e^{2\pi i x k/N}  $$

***

### The model

We evaluate the QFT of a comb distribution 

$$|\psi_0 \rangle = \frac{1}{\sqrt{2^{N-m}}}\sum_{t=0}^{2^{N-m}-1}|t 2^m\rangle~~, $$

 and compare the outcome population to the expected one


$$ |\psi_f \rangle = \frac{1}{\sqrt{2^{m}}}\sum_{t=0}^{2^m-1}|t 2^{n-m} \rangle~~.$$

We take the simple case of $m=0$, which results in the initial state

$$ |\psi_0 \rangle = \frac{1}{\sqrt{2^{n}}}\sum_{t=0}^{2^{n}-1}|t \rangle~~, $$

which in the qubit representation corresponds to 


$$|\psi_0\rangle =(|0\rangle + |1\rangle)\otimes(|0\rangle + |1\rangle)\otimes \cdots \otimes (|0\rangle + |1\rangle)  = H^{\otimes n} |0^n \rangle~~,$$

and the final state is given by 

$$|\psi_f \rangle = |0^n \rangle $$

The transformation is achieved by Classiq's built-in function `qft`, taking an argument `target`: `QArray[QBit]` a QFT is applied to `target`.


### The Scoring function
For a general comb state we define the score in terms of the **total variation distance** $D_{TV}(p,u) = \frac{1}{2}\sum_{i=0}^{N-1}\left|p_i-u_i \right|$, where $p_i$ and $u_i$ are the observed and expected probabilities, correspondingly. For the comb state we have $u_{t 2^m} = 1/m$ for $t=[0,2^{m}-1]$ and zero for the rest of the probabilities. The distance is bounded between zero and $1-1/n$. To obtain a score between zero and one (one is a perfect result) we define the score as
$$S =1 - \frac{D_{TV}(p,u)}{1-1/n} ~~.$$

***
***

## Defining a `BenchmarkExample` for QFT

In [1]:
import asyncio
import datetime
import sys
from pathlib import Path

sys.path.insert(0, "..")

import numpy as np
from benchmark import BenchmarkExample
from collector import ResultCollector
from hardware import HardwareRunner
from hardwares_preferences import HARDWARES, print_all_hardwares

from classiq import *

In [2]:
QFT_DESCRIPTION = Path("../descriptions/qft.tex").read_text(encoding="utf-8")

In [3]:
class QFTExample(BenchmarkExample):
    def __init__(self, num_qubits: int, m: int):
        super().__init__(
            name="qft",
            num_qubits=num_qubits,
            report_family_title="QFT",
            report_family_description=QFT_DESCRIPTION,
        )
        assert m < num_qubits
        self.m = m

    def create_main(self) -> callable:
        @qfunc
        def prepare_comb(m: CInt, qba: QArray):
            hadamard_transform(qba[m : qba.len])

        @qfunc
        def main(x: Output[QNum[self.num_qubits]]):
            allocate(x)
            prepare_comb(self.m, x)  # prepare the initial state, a comb state
            qft(x)

        return main

    async def submit(self, qprog: QuantumProgram) -> str:
        with ExecutionSession(qprog) as es:
            job = es.submit_sample()
            return job.id

    async def score(self, job_id):
        job = ExecutionJob.from_id(job_id)
        result = await job.result_async()
        df = result[0].value.dataframe

        n = self.num_qubits
        N = 2**n

        p = np.zeros(N)
        p[df["x"]] = df["probability"].to_numpy()

        u = np.zeros(N)
        if self.m == 0:
            u[0] = 1.0
        else:
            step = 2 ** (n - self.m)
            u[np.arange(0, N, step)] = 1.0 / (2**self.m)

        score = 1.0 - 0.5 * np.abs(p - u).sum() / (1.0 - 1.0 / n)

        exec_minutes = (job.end_time - job.start_time).total_seconds() / 60.0

        return {
            "score": float(score),
            "execution_time": exec_minutes,
        }

***
***
## Benchmarking a 4-qubits QFT

Define a specific example on 4 qubits:

In [4]:
example = QFTExample(num_qubits=4, m=0)
example.show()

Quantum program link: https://platform.classiq.io/circuit/3BS8LJDe6xE2aNlf9LvRne68RlX


### Set a `ResultCollector` with a file name fixed for the current example

In [5]:
FILENAME = example.default_results_filename

In [6]:
collector = ResultCollector(FILENAME, build_each_time=True)

### Choose on which backend to run 

We can print the list of backends:

In [7]:
print_all_hardwares(HARDWARES)

IBM Quantum: ibm_kingston, ibm_boston, ibm_marrakesh, ibm_torino, ibm_fez, ibm_pittsburg
IonQ: qpu.forte-1, qpu.forte-enterprise-1, qpu.forte-enterprise-2
Classiq: simulator, simulator_statevector, simulator_density_matrix, nvidia_simulator
Amazon Braket: Ankaa-3, Garnet, Forte 1
Azure Quantum: ionq.qpu.forte-enterprise-1, ionq.qpu.aria-1, ionq.qpu.forte-1


Here we choose one machine for IBM, one for IonQ, as well as Classiq simulators for reference.

*(The list of runners can be edited and added to the benchmarking run via the `ResultCollector`.)*

In [8]:
benchmark_hardware = [
    {"provider": "Classiq", "backend": "simulator"},
    {"provider": "Classiq", "backend": "simulator_statevector"},
    # {"provider": "IonQ", "backend": "qpu.forte-1"},
    # {
    #     "provider": "IBM Quantum",
    #     "backend": "ibm_kingston",
    #     "backend_kwargs": {
    #         "access_token": "PUT YOUR TOKEN HERE",
    #         "channel": "PUT YOUR CHANNEL HERE",
    #         "instance_crn": "PUT YOUR INSTANCE_CRN HERE",
    #     },
    # },
]

Define `HardwareRunner` for each backend, together with the number of shots and other hyperparameters:

In [9]:
common_config = {
    "max_timeout": 5e5,  # value is in seconds. Equals a bit more than 5 days."
    "num_shots": 1000,
}

In [10]:
runners = [
    HardwareRunner(
        cfg["provider"],
        cfg["backend"],
        **common_config,
        **(
            {"backend_kwargs": cfg["backend_kwargs"]} if "backend_kwargs" in cfg else {}
        ),
    )
    for cfg in benchmark_hardware
]

### Run Benchmark

In [11]:
print(
    "=" * 10
    + f"  {example.name}-{example.num_qubits} ({datetime.datetime.now()})   "
    + "=" * 10
)
await asyncio.gather(*[collector.run(runner, example) for runner in runners]);

==========  qft-4 (2026-03-25 22:37:34.814285)   ==========
2026-03-25 22:37:37.337917: Submit qft-4 for Classiq - simulator_statevector
2026-03-25 22:37:38.712989: Completed qft-4 for Classiq - simulator_statevector --> Score {'score': 0.9999999999999999, 'execution_time': 0.012155566666666668}
** Report updated: qft-4 for Classiq - simulator_statevector


In [12]:
await collector.print_status()

========== (2026-03-25 22:37:39.798661)   ==========
qft-4 | IonQ - qpu.forte-1 | COMPLETED | score=0.9973 | time=15.15 min
qft-4 | IBM Quantum - ibm_kingston | COMPLETED | score=0.6853 | time=86.55 min
qft-4 | Classiq - simulator | COMPLETED | score=1.0000 | time=0.01 min
qft-4 | Classiq - simulator_statevector | COMPLETED | score=1.0000 | time=0.01 min
